In [2]:
import requests

# Step 1: Get WFO + grid coords for Los Angeles lat/long
lat, lon = 34.05, -118.25
point_url = f"https://api.weather.gov/points/{lat},{lon}"
point_data = requests.get(point_url).json()

grid_id = point_data["properties"]["gridId"]      # e.g., "LOX"
grid_x  = point_data["properties"]["gridX"]       # e.g., 153
grid_y  = point_data["properties"]["gridY"]       # e.g., 44

# Step 2: Get 7-day forecast
forecast_url = f"https://api.weather.gov/gridpoints/{grid_id}/{grid_x},{grid_y}/forecast"
forecast_data = requests.get(forecast_url).json()

# Extract periods (each is ~12 hours: Day/Night)
for period in forecast_data["properties"]["periods"]:
    print(period["name"], period["temperature"], period["windSpeed"], period["shortForecast"])


Overnight 67 0 mph Patchy Fog
Tuesday 91 0 to 10 mph Patchy Fog then Mostly Sunny
Tuesday Night 68 0 to 5 mph Partly Cloudy
Wednesday 89 0 to 10 mph Mostly Cloudy then Slight Chance Rain Showers
Wednesday Night 69 5 to 10 mph Showers And Thunderstorms Likely
Thursday 84 5 to 10 mph Showers And Thunderstorms Likely
Thursday Night 66 0 to 10 mph Chance Showers And Thunderstorms
Friday 80 0 to 10 mph Chance Showers And Thunderstorms
Friday Night 66 0 to 10 mph Partly Cloudy then Patchy Fog
Saturday 83 0 to 10 mph Patchy Fog then Mostly Sunny
Saturday Night 66 0 to 10 mph Mostly Clear
Sunday 84 0 to 10 mph Sunny
Sunday Night 67 5 to 10 mph Patchy Fog
Monday 87 5 to 10 mph Patchy Fog then Mostly Sunny


In [3]:
# import requests

# Step 1: Get WFO + grid coords for Los Angeles lat/long
lat, lon = 34.05, -118.25
point_url = f"https://api.weather.gov/points/{lat},{lon}"
point_data = requests.get(point_url).json()

grid_id = point_data["properties"]["gridId"]      # e.g., "LOX"
grid_x  = point_data["properties"]["gridX"]       # e.g., 153
grid_y  = point_data["properties"]["gridY"]       # e.g., 44

# Step 2: Get 7-day forecast (Day + Night)
forecast_url = f"https://api.weather.gov/gridpoints/{grid_id}/{grid_x},{grid_y}/forecast"
forecast_data = requests.get(forecast_url).json()

# Step 3: Keep only "daytime" periods (skip nights)
daily_forecasts = [p for p in forecast_data["properties"]["periods"] if p["isDaytime"]]

# Step 4: Print clean daily data
for period in daily_forecasts:
    print(period["name"], period["temperature"], period["windSpeed"], period["shortForecast"])


Tuesday 91 0 to 10 mph Patchy Fog then Mostly Sunny
Wednesday 89 0 to 10 mph Mostly Cloudy then Slight Chance Rain Showers
Thursday 84 5 to 10 mph Showers And Thunderstorms Likely
Friday 80 0 to 10 mph Chance Showers And Thunderstorms
Saturday 83 0 to 10 mph Patchy Fog then Mostly Sunny
Sunday 84 0 to 10 mph Sunny
Monday 87 5 to 10 mph Patchy Fog then Mostly Sunny


In [4]:
import requests
import pandas as pd
import re

# Step 1: Get WFO + grid coords for Los Angeles lat/long
lat, lon = 34.05, -118.25
point_url = f"https://api.weather.gov/points/{lat},{lon}"
point_data = requests.get(point_url).json()

grid_id = point_data["properties"]["gridId"]      # e.g., "LOX"
grid_x  = point_data["properties"]["gridX"]       # e.g., 153
grid_y  = point_data["properties"]["gridY"]       # e.g., 44

# Step 2: Get 7-day forecast (Day + Night)
forecast_url = f"https://api.weather.gov/gridpoints/{grid_id}/{grid_x},{grid_y}/forecast"
forecast_data = requests.get(forecast_url).json()

# Step 3: Keep only "daytime" periods (skip nights)
daily_forecasts = [p for p in forecast_data["properties"]["periods"] if p["isDaytime"]]

# Step 4: Convert to DataFrame
df = pd.DataFrame([
    {
        "Date": pd.to_datetime(p["startTime"]).date(),
        "Day": p["name"],
        "Temp": p["temperature"],
        "Wind": p["windSpeed"],
        "Forecast": p["shortForecast"]
    }
    for p in daily_forecasts
])

# Optional: parse wind into average mph (numeric)
def parse_wind(wind_str):
    nums = re.findall(r"\d+", wind_str)
    if not nums:
        return None
    vals = list(map(int, nums))
    return sum(vals) / len(vals)  # average if "0 to 10 mph"

df["WindAvg"] = df["Wind"].apply(parse_wind)

# print(df)
df


,Date,Day,Temp,Wind,Forecast,WindAvg
0,2025-09-16,Tuesday,91,0 to 10 mph,Patchy Fog then Mostly Sunny,5.0
1,2025-09-17,Wednesday,89,0 to 10 mph,Mostly Cloudy then Slight Chance Rain Showers,5.0
2,2025-09-18,Thursday,84,5 to 10 mph,Showers And Thunderstorms Likely,7.5
3,2025-09-19,Friday,80,0 to 10 mph,Chance Showers And Thunderstorms,5.0
4,2025-09-20,Saturday,83,0 to 10 mph,Patchy Fog then Mostly Sunny,5.0
5,2025-09-21,Sunday,84,0 to 10 mph,Sunny,5.0
6,2025-09-22,Monday,87,5 to 10 mph,Patchy Fog then Mostly Sunny,7.5


In [6]:
# Create binary features from forecast text
df["ForecastLower"] = df["Forecast"].str.lower()

df["Fog"]     = df["ForecastLower"].apply(lambda x: 1 if "fog" in x else 0)
df["Rain"]    = df["ForecastLower"].apply(lambda x: 1 if "rain" in x or "showers" in x else 0)
df["Thunder"] = df["ForecastLower"].apply(lambda x: 1 if "thunder" in x else 0)
df["Cloud"]   = df["ForecastLower"].apply(lambda x: 1 if "cloud" in x else 0)
df["Sunny"]   = df["ForecastLower"].apply(lambda x: 1 if "sunny" in x else 0)

# Drop helper column
df = df.drop(columns=["ForecastLower"])

df


,Date,Day,Temp,Wind,Forecast,WindAvg,Fog,Rain,Thunder,Cloud,Sunny
0,2025-09-16,Tuesday,91,0 to 10 mph,Patchy Fog then Mostly Sunny,5.0,1,0,0,0,1
1,2025-09-17,Wednesday,89,0 to 10 mph,Mostly Cloudy then Slight Chance Rain Showers,5.0,0,1,0,1,0
2,2025-09-18,Thursday,84,5 to 10 mph,Showers And Thunderstorms Likely,7.5,0,1,1,0,0
3,2025-09-19,Friday,80,0 to 10 mph,Chance Showers And Thunderstorms,5.0,0,1,1,0,0
4,2025-09-20,Saturday,83,0 to 10 mph,Patchy Fog then Mostly Sunny,5.0,1,0,0,0,1
5,2025-09-21,Sunday,84,0 to 10 mph,Sunny,5.0,0,0,0,0,1
6,2025-09-22,Monday,87,5 to 10 mph,Patchy Fog then Mostly Sunny,7.5,1,0,0,0,1


In [11]:
# store the df to a csv file
df.to_csv("weather_forecast.csv", index=False)

### second code day #2

In [10]:
import requests
import pandas as pd

# Example Defining Site from your AQI dataset
# site_code = "06-037-1103"  # California, Los Angeles
site_code = "01-003-0010"
state, county, site = site_code.split("-")

# Your EPA AQS API credentials (replace with yours)
EMAIL = "deetkhan321@gmail.com"
API_KEY = "carmelfrog85"

# Query AQS site metadata
url = (
    f"https://aqs.epa.gov/data/api/list/sitesByCounty"
    f"?email={EMAIL}&key={API_KEY}&state={state}&county={county}"
)
resp = requests.get(url).json()

print(resp)

# Convert to DataFrame
# sites_df = pd.DataFrame(resp["Data"])

# sites_df

# # Find the site you care about
# site_info = sites_df[sites_df["site_number"] == site]

# print(site_info[["site_number", "site_name", "latitude", "longitude"]])


{'Header': [{'status': 'Success', 'request_time': '2025-09-17T03:12:33-04:00', 'url': 'https://aqs.epa.gov/data/api/list/sitesByCounty?email=deetkhan321@gmail.com&key=carmelfrog85&state=01&county=003', 'rows': 4}], 'Data': [{'code': '0001', 'value_represented': None}, {'code': '0002', 'value_represented': None}, {'code': '0003', 'value_represented': None}, {'code': '0010', 'value_represented': 'FAIRHOPE, Alabama'}]}


## Fetching back the lat , long based on country

In [9]:
import requests
import pandas as pd

EMAIL = "deetkhan321@gmail.com"
API_KEY = "carmelfrog85"

state = "01"   # Alabama
county = "003" # Baldwin
site = "0010"  # FAIRHOPE site
param = "88101"  # PM2.5 (you can also use 44201 for Ozone, etc.)
bdate = "20240101"
edate = "20241231"

# url = (
#     f"https://aqs.epa.gov/data/api/monitors/byCounty?"
#     f"email={EMAIL}&key={API_KEY}"
#     f"&param={param}&bdate={bdate}&edate={edate}"
#     f"&state={state}&county={county}&site={site}"
# )

url = (
    f"https://aqs.epa.gov/data/api/monitors/byCounty?"
    f"email={EMAIL}&key={API_KEY}"
    f"&param={param}&bdate={bdate}&edate={edate}"
    f"&state={state}&county={county}"
)

resp = requests.get(url).json()
print(resp)



{'Header': [{'status': 'Success', 'request_time': '2025-09-17T03:34:26-04:00', 'url': 'https://aqs.epa.gov/data/api/monitors/byCounty?email=deetkhan321@gmail.com&key=carmelfrog85&param=88101&bdate=20240101&edate=20241231&state=01&county=003', 'rows': 1}], 'Data': [{'state_code': '01', 'county_code': '003', 'site_number': '0010', 'parameter_code': '88101', 'poc': 3, 'parameter_name': 'PM2.5 - Local Conditions', 'open_date': '2023-01-01', 'close_date': None, 'concurred_exclusions': None, 'dominant_source': 'POINT', 'measurement_scale': 'NEIGHBORHOOD', 'measurement_scale_def': '500 M TO 4KM', 'monitoring_objective': 'POPULATION EXPOSURE', 'last_method_code': '209', 'last_method_description': 'Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM2.5C - Beta Attenuation', 'last_method_begin_date': '2023-01-01', 'naaqs_primary_monitor': 'Y', 'qa_primary_monitor': None, 'monitor_type': 'SLAMS', 'networks': None, 'monitoring_agency_code': '0013', 'monitoring_agency': 'Al Dept Of Env Mgt', 'si_id': 7,